<a href="https://colab.research.google.com/github/SahanaAbeysinghe/scam_baiter_active_defense_framework/blob/main/Autonomas_Scam_email_Detection_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import pandas as pd
import re
import io
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

print("Please upload your 'phishing_and_legitimate_emails.csv' dataset...")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[file_name]))

text_column = 'text'
label_column = 'label'

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text) # Remove HTML
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    stop_words = set(['the', 'is', 'in', 'and', 'to', 'a', 'of', 'for', 'it', 'on', 'that', 'this', 'with', 'as'])
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

print("Cleaning text data... this might take a moment.")
df['cleaned_text'] = df[text_column].apply(clean_text)

print("Applying TF-IDF Vectorization...")
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned_text'])
y = df[label_column]

joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Text cleaned, vectorized, and 'tfidf_vectorizer.pkl' saved.")

Please upload your 'phishing_and_legitimate_emails.csv' dataset...


Saving phishing_legit_dataset_KD_10000.csv to phishing_legit_dataset_KD_10000.csv
Cleaning text data... this might take a moment.
Applying TF-IDF Vectorization...
Text cleaned, vectorized, and 'tfidf_vectorizer.pkl' saved.


Training models

In [2]:
from sklearn.model_selection import train_test_split
import joblib

print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Splitting data into training and testing sets...


In [3]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

print("Training Multinomial Naive Bayes...")
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, nb_pred) * 100:.2f}%")
print(classification_report(y_test, nb_pred))

joblib.dump(nb_model, 'naive_bayes_detector.pkl')

Training Multinomial Naive Bayes...

Accuracy: 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       779
           1       1.00      1.00      1.00      1221

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



['naive_bayes_detector.pkl']

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, lr_pred) * 100:.2f}%")
print(classification_report(y_test, lr_pred))

joblib.dump(lr_model, 'logistic_regression_detector.pkl')

Training Logistic Regression...

Accuracy: 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       779
           1       1.00      1.00      1.00      1221

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



['logistic_regression_detector.pkl']

In [5]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

print("Training LinearSVC...")
svc_model = LinearSVC(max_iter=2000)
svc_model.fit(X_train, y_train)

svc_pred = svc_model.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, svc_pred) * 100:.2f}%")
print(classification_report(y_test, svc_pred))

joblib.dump(svc_model, 'linearsvc_detector.pkl')

Training LinearSVC...

Accuracy: 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       779
           1       1.00      1.00      1.00      1221

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



['linearsvc_detector.pkl']

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, rf_pred) * 100:.2f}%")
print(classification_report(y_test, rf_pred))

joblib.dump(rf_model, 'random_forest_detector.pkl')

Training Random Forest Classifier...

Accuracy: 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       779
           1       1.00      1.00      1.00      1221

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



['random_forest_detector.pkl']

In [7]:
import os
import time
import pandas as pd
import joblib


model_files = [
    'naive_bayes_detector.pkl',
    'logistic_regression_detector.pkl',
    'linearsvc_detector.pkl',
    'random_forest_detector.pkl'
]

proof_data = []

print("Running efficiency diagnostics...\n")

for file in model_files:

    model_name = file.split('_detector')[0].replace('_', ' ').title()

    size_kb = os.path.getsize(file) / 1024

    model = joblib.load(file)

    start_time = time.time()
    model.predict(X_test)  # Predicting on the entire test set
    end_time = time.time()

    speed_ms = (end_time - start_time) * 1000

    proof_data.append({
        "Model": model_name,
        "Accuracy": "100.0%",
        "File Size (KB)": round(size_kb, 2),
        "Prediction Time (ms)": round(speed_ms, 2)
    })

proof_df = pd.DataFrame(proof_data).sort_values(by="Prediction Time (ms)")
print("MODEL EFFICIENCY PROOF")
print(proof_df.to_string(index=False))

Running efficiency diagnostics...

MODEL EFFICIENCY PROOF
              Model Accuracy  File Size (KB)  Prediction Time (ms)
          Linearsvc   100.0%            5.67                  0.43
Logistic Regression   100.0%            5.80                  0.51
        Naive Bayes   100.0%           20.58                  1.98
      Random Forest   100.0%          175.63                 24.50
